# Context-aware XGBoost for masked vertebral morphology

This notebook demonstrates a proof of concept for **frontal AP/PA radiographs**. Given the morphology of vertebrae at relative positions `-2`, `-1`, `+1`, and `+2`, the model estimates the expected morphology of a masked target vertebra.

The notebook uses synthetic four-corner landmarks so it can run without clinical data. Replace the synthetic table with a real landmark CSV using the adapter near the end.

> Research screening-support demonstration only. The predictions are expected geometric measurements, not diagnoses, and require clinical validation and interpretation.

## Demonstrated flow

```text
TL, TR, BL, BR landmarks for each vertebra
                 ↓
Frontal morphology measurements
                 ↓
Mask one target vertebra
                 ↓
Normalize using visible neighbors only
                 ↓
Neighbor interpolation baseline
                 ↓
XGBoost predicts residual corrections
                 ↓
Patient-grouped evaluation on unseen patients
```

Canonical corner order is **TL, TR, BL, BR**, with `(x, y)` in image-pixel coordinates. Left and right mean **patient-left and patient-right** after image orientation has been standardized.

## Setup

Uncomment and run the next line once if the notebook environment does not already contain the tabular-model dependencies.

In [ ]:
# %pip install -q "xgboost>=2.0" "scikit-learn>=1.3" pandas matplotlib

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRegressor

SEED = 20260812
RNG = np.random.default_rng(SEED)

LEVELS = ["T10", "T11", "T12", "L1", "L2", "L3", "L4", "L5"]
OFFSETS = (-2, -1, 1, 2)

pd.set_option("display.max_columns", 100)
print("Ready. XGBoost will run on CPU with deterministic seeds.")

## 1. Create a synthetic AP/PA landmark table

The simulation includes patient-specific size, study magnification, smooth level-dependent anatomy, coronal curvature, and landmark noise. It exists only to demonstrate the pipeline; its scores are not evidence of clinical performance. Each simulated patient has two radiographs so patient-level grouping can be checked explicitly.

In [ ]:
def rotate_points(points, angle_deg):
    angle = np.deg2rad(angle_deg)
    rotation = np.array([[np.cos(angle), -np.sin(angle)],
                         [np.sin(angle),  np.cos(angle)]])
    return np.asarray(points, dtype=float) @ rotation.T


def corners_from_morphology(center_x, center_y, superior_width, inferior_width,
                            left_height, right_height, orientation_deg):
    """Return corners in canonical TL, TR, BL, BR order."""
    local = np.array([
        [-superior_width / 2, -left_height / 2],
        [ superior_width / 2, -right_height / 2],
        [-inferior_width / 2,  left_height / 2],
        [ inferior_width / 2,  right_height / 2],
    ])
    return rotate_points(local, orientation_deg) + np.array([center_x, center_y])


def simulate_landmarks(n_patients=180, images_per_patient=2, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    base_heights = np.array([42, 43, 45, 47, 49, 51, 52, 51], dtype=float)
    base_widths = np.array([72, 75, 79, 84, 89, 94, 98, 100], dtype=float)

    for patient_number in range(n_patients):
        patient_id = f"P{patient_number:04d}"
        height_factor = rng.lognormal(mean=0.0, sigma=0.075)
        width_factor = rng.lognormal(mean=0.0, sigma=0.085)
        patient_asymmetry = rng.normal(0.0, 0.018)
        patient_shape = rng.normal(0.0, 0.025)

        for study_number in range(images_per_patient):
            image_id = f"{patient_id}_IMG{study_number + 1}"
            view = rng.choice(["AP", "PA"])
            magnification = rng.lognormal(mean=0.0, sigma=0.055)
            curve_amplitude = rng.normal(0.0, 24.0)
            curve_phase = rng.uniform(-0.6, 0.6)
            image_rotation = rng.normal(0.0, 1.5)
            y_start = 260 + rng.normal(0.0, 8.0)

            for level_index, level in enumerate(LEVELS):
                progress = level_index / (len(LEVELS) - 1)
                smooth_level_effect = np.sin(progress * np.pi)
                mean_height = (base_heights[level_index] * height_factor * magnification
                               * (1 + patient_shape * smooth_level_effect))
                mean_width = (base_widths[level_index] * width_factor * magnification
                              * (1 - 0.5 * patient_shape * smooth_level_effect))

                asymmetry = patient_asymmetry + 0.010 * np.sin(level_index + curve_phase)
                left_height = mean_height * (1 + asymmetry) * rng.normal(1.0, 0.009)
                right_height = mean_height * (1 - asymmetry) * rng.normal(1.0, 0.009)
                superior_width = mean_width * rng.normal(0.985, 0.008)
                inferior_width = mean_width * rng.normal(1.015, 0.008)

                phase = progress * np.pi + curve_phase
                center_x = 500 + curve_amplitude * np.sin(phase) + rng.normal(0.0, 1.2)
                center_y = y_start + level_index * 116 * magnification + rng.normal(0.0, 1.5)
                local_tilt = image_rotation + 0.10 * curve_amplitude * np.cos(phase)
                corners = corners_from_morphology(
                    center_x, center_y, superior_width, inferior_width,
                    left_height, right_height, local_tilt
                )
                corners += rng.normal(0.0, 0.45, size=corners.shape)

                row = {
                    "patient_id": patient_id,
                    "image_id": image_id,
                    "view": view,
                    "vertebral_level": level,
                    "target_eligible": True,
                    "image_width": 1000,
                    "image_height": 1400,
                }
                for name, point in zip(("tl", "tr", "bl", "br"), corners):
                    row[f"{name}_x"] = float(point[0])
                    row[f"{name}_y"] = float(point[1])
                rows.append(row)

    return pd.DataFrame(rows)


landmarks = simulate_landmarks()
print(f"{landmarks['patient_id'].nunique():,} patients")
print(f"{landmarks['image_id'].nunique():,} images")
print(f"{len(landmarks):,} vertebra rows")
display(landmarks.head())

In [ ]:
LANDMARK_COLUMNS = [
    "tl_x", "tl_y", "tr_x", "tr_y",
    "bl_x", "bl_y", "br_x", "br_y",
]
IDENTIFIER_COLUMNS = ["patient_id", "image_id", "vertebral_level", "view"]


def validate_landmark_table(frame):
    required = IDENTIFIER_COLUMNS + LANDMARK_COLUMNS
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if frame[required].isna().any().any():
        raise ValueError("Required identifiers and landmark coordinates cannot be missing.")
    if frame.duplicated(["image_id", "vertebral_level"]).any():
        raise ValueError("Each image may contain only one row per vertebral level.")
    coordinates = frame[LANDMARK_COLUMNS].to_numpy(dtype=float)
    if not np.isfinite(coordinates).all():
        raise ValueError("Landmark coordinates must be finite.")
    unknown_levels = sorted(set(frame["vertebral_level"]) - set(LEVELS))
    if unknown_levels:
        raise ValueError(f"Unexpected vertebral levels: {unknown_levels}")
    return frame.copy()


landmarks = validate_landmark_table(landmarks)
print("Landmark schema and image/level uniqueness checks passed.")

In [ ]:
def row_corners(row):
    return np.array([
        [row.tl_x, row.tl_y], [row.tr_x, row.tr_y],
        [row.bl_x, row.bl_y], [row.br_x, row.br_y],
    ], dtype=float)


def plot_spine(frame, image_id, masked_level=None, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 8))
    image_rows = frame.loc[frame["image_id"] == image_id].copy()
    image_rows["level_order"] = image_rows["vertebral_level"].map(LEVELS.index)
    image_rows = image_rows.sort_values("level_order")
    for row in image_rows.itertuples():
        corners = row_corners(row)
        polygon_order = corners[[0, 1, 3, 2]]
        is_masked = row.vertebral_level == masked_level
        patch = Polygon(
            polygon_order, closed=True,
            facecolor="white" if is_masked else "#8ecae6",
            edgecolor="#d62728" if is_masked else "#023047",
            hatch="///" if is_masked else None, alpha=0.9, linewidth=2,
        )
        ax.add_patch(patch)
        ax.text(corners[:, 0].mean(), corners[:, 1].mean(), row.vertebral_level,
                ha="center", va="center", fontsize=9)
    ax.set_xlim(350, 650)
    ax.set_ylim(1150, 150)
    ax.set_aspect("equal")
    ax.set_title(f"Synthetic frontal spine: {image_id}")
    ax.set_xlabel("x (pixels; patient-left to patient-right)")
    ax.set_ylabel("y (pixels)")
    return ax


example_image = landmarks["image_id"].iloc[0]
plot_spine(landmarks, example_image, masked_level="L2")
plt.show()

## 2. Convert four corners into frontal morphology

Lengths use Euclidean distance, so modest coronal tilt does not shorten them. `orientation_deg` is the axial/circular mean of the superior and inferior endplate angles. Angles are wrapped to `[-90°, 90°)` because a line has 180-degree symmetry.

In [ ]:
def point_distance(x1, y1, x2, y2):
    return np.hypot(np.asarray(x2) - np.asarray(x1), np.asarray(y2) - np.asarray(y1))


def line_angle_deg(x1, y1, x2, y2):
    return np.rad2deg(np.arctan2(np.asarray(y2) - np.asarray(y1),
                                  np.asarray(x2) - np.asarray(x1)))


def wrap_axial_deg(angle):
    return (np.asarray(angle) + 90.0) % 180.0 - 90.0


def axial_mean_deg(first, second):
    first2 = np.deg2rad(2.0 * np.asarray(first))
    second2 = np.deg2rad(2.0 * np.asarray(second))
    mean = 0.5 * np.rad2deg(np.arctan2(
        np.sin(first2) + np.sin(second2),
        np.cos(first2) + np.cos(second2),
    ))
    return wrap_axial_deg(mean)


def calculate_frontal_morphology(frame):
    result = frame.copy()
    result["superior_width"] = point_distance(
        result.tl_x, result.tl_y, result.tr_x, result.tr_y)
    result["inferior_width"] = point_distance(
        result.bl_x, result.bl_y, result.br_x, result.br_y)
    result["left_height"] = point_distance(
        result.tl_x, result.tl_y, result.bl_x, result.bl_y)
    result["right_height"] = point_distance(
        result.tr_x, result.tr_y, result.br_x, result.br_y)
    result["mean_width"] = 0.5 * (result.superior_width + result.inferior_width)
    result["mean_height"] = 0.5 * (result.left_height + result.right_height)
    result["left_right_height_ratio"] = result.left_height / result.right_height
    result["width_height_ratio"] = result.mean_width / result.mean_height
    result["superior_endplate_angle_deg"] = line_angle_deg(
        result.tl_x, result.tl_y, result.tr_x, result.tr_y)
    result["inferior_endplate_angle_deg"] = line_angle_deg(
        result.bl_x, result.bl_y, result.br_x, result.br_y)
    result["orientation_deg"] = axial_mean_deg(
        result.superior_endplate_angle_deg, result.inferior_endplate_angle_deg)
    result["endplate_nonparallel_deg"] = np.abs(wrap_axial_deg(
        result.superior_endplate_angle_deg - result.inferior_endplate_angle_deg))
    result["center_x"] = result[["tl_x", "tr_x", "bl_x", "br_x"]].mean(axis=1)
    result["center_y"] = result[["tl_y", "tr_y", "bl_y", "br_y"]].mean(axis=1)

    positive = ["superior_width", "inferior_width", "left_height", "right_height"]
    if (result[positive] <= 0).any().any():
        raise ValueError("All vertebral dimensions must be positive.")
    return result


morphology = calculate_frontal_morphology(landmarks)
display(morphology[[
    "patient_id", "image_id", "vertebral_level",
    "left_height", "right_height", "superior_width",
    "inferior_width", "orientation_deg"
]].head())

## 3. Construct leakage-safe masked samples

For every eligible target with all four required neighbors:

- dimensions are normalized using medians from `-2`, `-1`, `+1`, and `+2` only;
- the baseline is the average morphology of `-1` and `+1`;
- the orientation baseline is their axial mean;
- XGBoost learns the target-minus-baseline residual;
- target landmarks and target morphology never enter `X`.

This layout trains one model across levels while retaining `target_level` as anatomical context.

In [ ]:
DIMENSIONS = ("left_height", "right_height", "superior_width", "inferior_width")
NEIGHBOR_FEATURES = (
    "left_height_norm", "right_height_norm",
    "superior_width_norm", "inferior_width_norm",
    "left_right_height_ratio", "width_height_ratio",
    "orientation_relative_deg", "center_dx_norm", "center_dy_norm",
)


def offset_name(offset):
    return f"m{abs(offset)}" if offset < 0 else f"p{offset}"


def construct_masked_samples(morphology_frame):
    records = []
    for image_id, image_rows in morphology_frame.groupby("image_id", sort=False):
        image_rows = image_rows.set_index("vertebral_level", drop=False)
        patient_ids = image_rows["patient_id"].unique()
        if len(patient_ids) != 1:
            raise ValueError(f"Image {image_id} maps to more than one patient.")

        for target_position in range(2, len(LEVELS) - 2):
            target_level = LEVELS[target_position]
            required_levels = [LEVELS[target_position + offset] for offset in OFFSETS]
            if target_level not in image_rows.index or not set(required_levels).issubset(image_rows.index):
                continue

            target = image_rows.loc[target_level]
            if "target_eligible" in image_rows.columns and not bool(target["target_eligible"]):
                continue
            neighbors = {
                offset: image_rows.loc[LEVELS[target_position + offset]]
                for offset in OFFSETS
            }
            ref_height = float(np.median([row.mean_height for row in neighbors.values()]))
            ref_width = float(np.median([row.mean_width for row in neighbors.values()]))
            if ref_height <= 0 or ref_width <= 0:
                continue

            nearest_superior = neighbors[-1]
            nearest_inferior = neighbors[1]
            baseline_orientation = float(axial_mean_deg(
                nearest_superior.orientation_deg, nearest_inferior.orientation_deg))
            anchor_x = 0.5 * (nearest_superior.center_x + nearest_inferior.center_x)
            anchor_y = 0.5 * (nearest_superior.center_y + nearest_inferior.center_y)

            record = {
                "patient_id": patient_ids[0],
                "image_id": image_id,
                "view": target["view"],
                "target_level": target_level,
                "target_level_code": target_position,
                "ref_height": ref_height,
                "ref_width": ref_width,
                "anchor_x": anchor_x,
                "anchor_y": anchor_y,
                "baseline_orientation_deg": baseline_orientation,
            }

            for offset, neighbor in neighbors.items():
                prefix = offset_name(offset)
                record[f"{prefix}_left_height_norm"] = neighbor.left_height / ref_height
                record[f"{prefix}_right_height_norm"] = neighbor.right_height / ref_height
                record[f"{prefix}_superior_width_norm"] = neighbor.superior_width / ref_width
                record[f"{prefix}_inferior_width_norm"] = neighbor.inferior_width / ref_width
                record[f"{prefix}_left_right_height_ratio"] = neighbor.left_right_height_ratio
                record[f"{prefix}_width_height_ratio"] = neighbor.width_height_ratio
                record[f"{prefix}_orientation_relative_deg"] = float(wrap_axial_deg(
                    neighbor.orientation_deg - baseline_orientation))
                record[f"{prefix}_center_dx_norm"] = (neighbor.center_x - anchor_x) / ref_width
                record[f"{prefix}_center_dy_norm"] = (neighbor.center_y - anchor_y) / ref_height

            for dimension in DIMENSIONS:
                scale = ref_height if "height" in dimension else ref_width
                target_norm = target[dimension] / scale
                baseline_norm = 0.5 * (nearest_superior[dimension] + nearest_inferior[dimension]) / scale
                record[f"target_{dimension}_norm"] = target_norm
                record[f"baseline_{dimension}_norm"] = baseline_norm
                record[f"y_{dimension}_residual"] = target_norm - baseline_norm

            record["target_orientation_deg"] = target.orientation_deg
            record["y_orientation_residual_deg"] = float(wrap_axial_deg(
                target.orientation_deg - baseline_orientation))
            records.append(record)

    return pd.DataFrame.from_records(records)


samples = construct_masked_samples(morphology)
print(f"Created {len(samples):,} masked samples from {samples['image_id'].nunique():,} images.")
print("Targets per level:")
display(samples["target_level"].value_counts().sort_index())
display(samples.head(3))

In [ ]:
neighbor_columns = [
    f"{offset_name(offset)}_{feature}"
    for offset in OFFSETS
    for feature in NEIGHBOR_FEATURES
]
model_input = samples[neighbor_columns + ["target_level", "view"]].copy()
X = pd.get_dummies(model_input, columns=["target_level", "view"], dtype=float)

TARGET_COLUMNS = {
    "left_height": "y_left_height_residual",
    "right_height": "y_right_height_residual",
    "superior_width": "y_superior_width_residual",
    "inferior_width": "y_inferior_width_residual",
    "orientation": "y_orientation_residual_deg",
}

for forbidden in ("patient_id", "image_id"):
    assert forbidden not in X.columns
assert not any(column.startswith("target_left_height") for column in X.columns)
assert np.isfinite(X.to_numpy(dtype=float)).all()
assert np.isfinite(samples[list(TARGET_COLUMNS.values())].to_numpy(dtype=float)).all()

print(f"Model matrix: {X.shape[0]:,} samples × {X.shape[1]:,} features")
print("Leakage audit passed: identifiers and masked target morphology are absent from X.")

## 4. Split by patient

All samples and radiographs from one patient must stay in exactly one partition. The first split reserves 20% of patients for the final test set; a second grouped split creates a validation set from the remaining patients.

For real project data, preserve the repository's predefined patient-grouped train/validation/test assignments instead of re-splitting them.

In [ ]:
def grouped_train_val_test_indices(frame, test_size=0.20, val_size=0.20, seed=SEED):
    outer = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    dev_position, test_position = next(outer.split(frame, groups=frame.patient_id))
    dev = frame.iloc[dev_position]

    relative_val_size = val_size / (1.0 - test_size)
    inner = GroupShuffleSplit(n_splits=1, test_size=relative_val_size, random_state=seed + 1)
    train_position, val_position = next(inner.split(dev, groups=dev.patient_id))
    return dev.iloc[train_position].index, dev.iloc[val_position].index, frame.iloc[test_position].index


train_idx, val_idx, test_idx = grouped_train_val_test_indices(samples)
partitions = {"train": train_idx, "validation": val_idx, "test": test_idx}
patient_sets = {name: set(samples.loc[index, "patient_id"]) for name, index in partitions.items()}
assert patient_sets["train"].isdisjoint(patient_sets["validation"])
assert patient_sets["train"].isdisjoint(patient_sets["test"])
assert patient_sets["validation"].isdisjoint(patient_sets["test"])

split_summary = pd.DataFrame({
    name: {
        "patients": samples.loc[index, "patient_id"].nunique(),
        "images": samples.loc[index, "image_id"].nunique(),
        "masked_samples": len(index),
    }
    for name, index in partitions.items()
}).T
display(split_summary)
print("Patient overlap: none")

## 5. Train one XGBoost model per morphology output

Separate regressors keep output-specific error analysis straightforward. The validation partition is used for early stopping. Hyperparameters are intentionally modest for a quick CPU demonstration; tune them later with grouped cross-validation on the development patients only.

In [ ]:
models = {}
predicted_residuals = {}

for output_name, target_column in TARGET_COLUMNS.items():
    model = XGBRegressor(
        objective="reg:squarederror",
        eval_metric="mae",
        n_estimators=500,
        learning_rate=0.035,
        max_depth=3,
        min_child_weight=4,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        tree_method="hist",
        early_stopping_rounds=40,
        random_state=SEED,
        n_jobs=4,
    )
    model.fit(
        X.loc[train_idx], samples.loc[train_idx, target_column],
        eval_set=[(X.loc[val_idx], samples.loc[val_idx, target_column])],
        verbose=False,
    )
    models[output_name] = model
    predicted_residuals[output_name] = model.predict(X.loc[test_idx])
    print(f"{output_name:>16}: best iteration = {model.best_iteration}")

## 6. Compare XGBoost with interpolation

Dimension errors are reported in neighbor-normalized units. Orientation error is the wrapped absolute angular difference in degrees. The critical proof-of-concept question is whether XGBoost improves on the simple `-1/+1` interpolation baseline for unseen patients.

In [ ]:
prediction_frame = samples.loc[test_idx, [
    "patient_id", "image_id", "target_level",
    "ref_height", "ref_width", "anchor_x", "anchor_y",
    "baseline_orientation_deg",
]].copy()
metric_rows = []

for output_name in TARGET_COLUMNS:
    if output_name == "orientation":
        actual = samples.loc[test_idx, "target_orientation_deg"].to_numpy()
        baseline = samples.loc[test_idx, "baseline_orientation_deg"].to_numpy()
        predicted = wrap_axial_deg(baseline + predicted_residuals[output_name])
        baseline_error = np.abs(wrap_axial_deg(actual - baseline))
        xgb_error = np.abs(wrap_axial_deg(actual - predicted))
        unit = "degrees"
    else:
        actual = samples.loc[test_idx, f"target_{output_name}_norm"].to_numpy()
        baseline = samples.loc[test_idx, f"baseline_{output_name}_norm"].to_numpy()
        predicted = baseline + predicted_residuals[output_name]
        baseline_error = np.abs(actual - baseline)
        xgb_error = np.abs(actual - predicted)
        unit = "neighbor-normalized"

    prediction_frame[f"{output_name}_actual"] = actual
    prediction_frame[f"{output_name}_baseline"] = baseline
    prediction_frame[f"{output_name}_xgb"] = predicted
    baseline_mae = float(np.mean(baseline_error))
    xgb_mae = float(np.mean(xgb_error))
    metric_rows.append({
        "output": output_name,
        "unit": unit,
        "baseline_mae": baseline_mae,
        "xgboost_mae": xgb_mae,
        "relative_improvement_pct": 100.0 * (baseline_mae - xgb_mae) / baseline_mae,
    })

metrics = pd.DataFrame(metric_rows).set_index("output")
display(metrics.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dimension_metrics = metrics.loc[["left_height", "right_height", "superior_width", "inferior_width"]]
dimension_metrics[["baseline_mae", "xgboost_mae"]].plot.bar(
    ax=axes[0], color=["#9aa0a6", "#219ebc"])
axes[0].set_title("Dimension MAE (lower is better)")
axes[0].set_ylabel("Neighbor-normalized MAE")
axes[0].tick_params(axis="x", rotation=30)

metrics.loc[["orientation"], ["baseline_mae", "xgboost_mae"]].plot.bar(
    ax=axes[1], color=["#9aa0a6", "#fb8500"])
axes[1].set_title("Orientation MAE (lower is better)")
axes[1].set_ylabel("Degrees")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
level_rows = []
for level, group in prediction_frame.groupby("target_level"):
    for output_name in TARGET_COLUMNS:
        if output_name == "orientation":
            error = np.abs(wrap_axial_deg(
                group[f"{output_name}_actual"] - group[f"{output_name}_xgb"]))
        else:
            error = np.abs(group[f"{output_name}_actual"] - group[f"{output_name}_xgb"])
        level_rows.append({"target_level": level, "output": output_name, "mae": error.mean()})

level_metrics = pd.DataFrame(level_rows).pivot(index="target_level", columns="output", values="mae")
display(level_metrics.reindex([level for level in LEVELS if level in level_metrics.index]).round(4))

In [ ]:
importance = pd.DataFrame({
    output_name: model.feature_importances_
    for output_name, model in models.items()
}, index=X.columns)
importance["mean_importance"] = importance.mean(axis=1)
top_importance = importance.sort_values("mean_importance", ascending=False).head(15)

ax = top_importance["mean_importance"].sort_values().plot.barh(
    figsize=(8, 5), color="#219ebc")
ax.set_title("Mean built-in feature importance across outputs")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

print("Use held-out permutation importance or SHAP for formal interpretation; built-in importance is exploratory.")

## 7. Visualize one masked reconstruction

The notebook predicts morphology, not target location. For visualization only, the target center is interpolated from the nearest superior and inferior centers. The solid orange quadrilateral is the XGBoost estimate; the dashed green quadrilateral shows the hidden synthetic target for comparison.

In [ ]:
example_prediction = prediction_frame.iloc[0]
target_level = example_prediction.target_level
image_id = example_prediction.image_id

predicted_corners = corners_from_morphology(
    example_prediction.anchor_x,
    example_prediction.anchor_y,
    example_prediction.superior_width_xgb * example_prediction.ref_width,
    example_prediction.inferior_width_xgb * example_prediction.ref_width,
    example_prediction.left_height_xgb * example_prediction.ref_height,
    example_prediction.right_height_xgb * example_prediction.ref_height,
    example_prediction.orientation_xgb,
)
true_row = landmarks.loc[
    (landmarks.image_id == image_id) & (landmarks.vertebral_level == target_level)
].iloc[0]
true_corners = row_corners(true_row)

fig, ax = plt.subplots(figsize=(6, 8))
plot_spine(landmarks.loc[landmarks.vertebral_level != target_level], image_id, ax=ax)
ax.add_patch(Polygon(true_corners[[0, 1, 3, 2]], closed=True, fill=False,
                     edgecolor="#2a9d8f", linestyle="--", linewidth=3, label="Hidden target"))
ax.add_patch(Polygon(predicted_corners[[0, 1, 3, 2]], closed=True, fill=False,
                     edgecolor="#fb8500", linewidth=3, label="XGBoost estimate"))
ax.legend(loc="lower right")
ax.set_title(f"Masked {target_level} reconstruction — {image_id}")
plt.show()

## 8. Replace synthetic data with real annotations

The minimal CSV schema is:

```text
patient_id,image_id,view,vertebral_level,
tl_x,tl_y,tr_x,tr_y,bl_x,bl_y,br_x,br_y
```

Optional but recommended columns include `target_eligible`, image dimensions, pixel spacing, source site, projection-quality flags, and landmark confidence. Set `target_eligible=False` for targets that should not define expected normal morphology. Neighbors can later be handled with explicit quality or missingness rules.

The current project COCO annotations contain ordered corners and a chain index, but not necessarily a verified anatomical vertebral level. Do **not** silently treat chain rank as `T10`, `T11`, ..., because cropping and visibility vary between images. Add or validate anatomical level labels before training a level-aware model.

In [ ]:
def load_real_landmarks(csv_path):
    frame = pd.read_csv(csv_path)
    if "target_eligible" not in frame.columns:
        warnings.warn(
            "target_eligible is absent; all vertebrae will be treated as eligible normal targets. "
            "Review this assumption before clinical experiments."
        )
        frame["target_eligible"] = True
    return validate_landmark_table(frame)


# Example switch to real data:
# real_csv = Path("path/to/frontal_vertebra_landmarks.csv")
# landmarks = load_real_landmarks(real_csv)
# morphology = calculate_frontal_morphology(landmarks)
# samples = construct_masked_samples(morphology)
# Then rerun the feature, split, training, and evaluation sections.

## Interpretation and next steps

A successful first experiment should show that XGBoost consistently beats neighbor interpolation on a locked, patient-grouped test set—not merely on these synthetic data. Before using observed-minus-expected residuals as a suspicious-morphology signal:

1. establish target eligibility using clinical review rather than geometry alone;
2. preserve predefined patient-grouped splits and add an external site/scanner test set;
3. report MAE and high-percentile errors separately by vertebral level, dataset, AP/PA view, and quality group;
4. quantify landmark annotation repeatability to estimate the achievable error floor;
5. add uncertainty intervals and an abstention rule for missing, rotated, instrumented, or poorly visualized context;
6. keep expected-morphology estimation separate from any clinically validated classification stage.